# Feature Engineering

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, make_scorer

In [10]:
PROCESSED_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data\processed"

preprocessor = joblib.load(os.path.join(PROCESSED_DIR, "preprocessor.joblib"))
X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze("columns")

In [12]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

X_train: (239, 12)
X_test: (60, 12)


In [19]:
LOG_COLS = ["creatinine_phosphokinase", "serum_creatinine", "platelets", "time"]
SCALE_COLS = ["age", "ejection_fraction", "serum_sodium"]
BIN_COLS = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]

def to_raw(X_df):
    X = X_df.copy()
    col_map = {f"log__{c}": c for c in LOG_COLS}
    col_map |= {f"scale__{c}": c for c in SCALE_COLS}
    col_map |= {f"pass__{c}": c for c in BIN_COLS}
    X = X.rename(columns=col_map)

    for c in LOG_COLS:
        X[c] = np.expm1(X[c])

    scaler = preprocessor.named_transformers_["scale"]
    for c, mu, sd in zip(SCALE_COLS, scaler.mean_, scaler.scale_):
        X[c] = X[c] * sd + mu
    return X

raw_train, raw_test = to_raw(X_train), to_raw(X_test)
print(raw_train[["age", "serum_creatinine", "sex"]].head())

    age  serum_creatinine  sex
0  58.0               1.0  0.0
1  53.0               0.8  1.0
2  75.0               1.9  1.0
3  64.0               2.4  1.0
4  45.0               1.6  1.0


In [20]:
raw_train

,creatinine_phosphokinase,serum_creatinine,platelets,time,age,ejection_fraction,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
0,400.0,1.0,164000.0,91.0,58.0,40.0,139.0,1.0,0.0,0.0,0.0,0.0
1,63.0,0.8,368000.0,22.0,53.0,60.0,135.0,0.0,1.0,0.0,1.0,0.0
2,582.0,1.9,265000.0,4.0,75.0,20.0,130.0,0.0,0.0,1.0,1.0,0.0
3,143.0,2.4,246000.0,214.0,64.0,25.0,135.0,0.0,0.0,0.0,1.0,0.0
4,582.0,1.6,126000.0,180.0,45.0,20.0,135.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
234,211.0,1.2,274000.0,207.0,72.0,25.0,134.0,0.0,0.0,0.0,0.0,0.0
235,1082.0,6.1,250000.0,107.0,60.0,45.0,131.0,1.0,1.0,0.0,1.0,0.0
236,69.0,1.0,132000.0,147.0,49.0,50.0,140.0,1.0,0.0,0.0,0.0,0.0
237,369.0,1.6,252000.0,90.0,50.0,25.0,136.0,0.0,1.0,0.0,1.0,0.0


In [25]:
def add_egfr(df_new):
    df_new = df_new.copy()
    scr, age, sex = df_new["serum_creatinine"], df_new["age"], df_new["sex"]
    k = np.where(sex == 0, 0.7, 0.9)
    alpha = np.where(sex == 0, -0.241, -0.302)
    df_new["egfr"] = (142 * np.minimum(scr / k, 1.0) ** alpha * np.maximum(scr / k, 1.0) ** -1.200 * 0.9938 ** age)
    return df_new

def add_ef_group(df_new):
    df_new = df_new.copy()
    df_new["ef_group"] = np.where(df_new["ejection_fraction"] < 40, 0, np.where(df_new["ejection_fraction"] <= 49, 1, 2))
    return df_new

In [26]:
raw_tr_fe = add_ef_group(add_egfr(raw_train))
raw_te_fe = add_ef_group(add_egfr(raw_test))
print("eGFR range :", raw_tr_fe["egfr"].min().round(1), "-", raw_tr_fe["egfr"].max().round(1))
print(raw_tr_fe["ef_group"].value_counts().sort_index())

eGFR range : 4.7 - 118.0
ef_group
0    144
1     47
2     48
Name: count, dtype: int64
